# Topic: ML: K-Nearest Neighbors (KNN) & Distance Metrics

## Definition (30-second explanation)
K-Nearest Neighbors (KNN) is a simple, non-parametric, lazy learning algorithm used for both classification and regression. It predicts the label or value of a new data point by finding the 'K' closest data points in the training set and taking a majority vote (for classification) or an average (for regression), using a specific distance metric to define "closeness."

## Why Interviewers Ask This
*   **Fundamental algorithm:** It tests your understanding of instance-based learning vs. model-based learning.
*   **Distance Metrics:** It's the standard entry point to discuss how to mathematically define similarity (Euclidean vs. Cosine vs. Manhattan).
*   **Curse of Dimensionality:** KNN is the classic algorithm used to test if you understand why distance breaks down in high-dimensional spaces.
*   **Production Awareness:** Interviewers want to see if you know *not* to use basic KNN for low-latency, large-scale production scoring.

## Core Concepts
*   **Lazy Learning:** No explicit training phase; the model *is* the training data. Computation happens entirely at inference time.
*   **Choice of K:** A small K (e.g., K=1) has low bias but high variance (overfitting, sensitive to noise). A large K has high bias but low variance (underfitting).
*   **Distance Metrics:** 
    *   *Euclidean:* Standard straight-line distance (L2 norm). Sensitive to scale.
    *   *Manhattan:* Grid-like path (L1 norm). Often better for high dimensions or categorical grids.
    *   *Cosine:* Measures the angle between vectors, ignoring magnitude. Great for text/TF-IDF.
    *   *Mahalanobis:* Accounts for covariance and scaling. Useful when features are correlated.
*   **Curse of Dimensionality:** In high dimensions, the distance between the nearest and farthest neighbors becomes almost equal, rendering "closeness" meaningless.

## When to Use
*   When a simple baseline model is needed quickly.
*   When the decision boundary is highly irregular and non-linear.
*   When interpretability is important (you can easily show *why* a prediction was made by showing the neighbors).
*   When the dataset is relatively small and feature dimensions are low.

## Advantages
*   No assumptions about the underlying data distribution (non-parametric).
*   Incredibly simple to implement and understand.
*   Naturally handles multi-class classification out of the box.
*   Can adapt immediately to new training data (just add it to the dataset).

## Limitations
*   **Computationally expensive at inference:** Must calculate distance to every single training point for every prediction $O(N \times D)$.
*   **Memory intensive:** Requires storing the entire dataset in memory.
*   Requires rigorous feature scaling (normalization/standardization); otherwise, large-scale features dominate the distance calculation.
*   Fails dramatically in high-dimensional spaces.

## Common Comparisons
*   **KNN vs. Decision Trees:** Trees are eager learners, handle unscaled data well, and do feature selection implicitly. KNN is lazy, needs scaled data, and uses all features equally (unless weighted).
*   **KNN vs. Logistic Regression:** Logistic regression assumes a linear decision boundary and outputs probabilities. KNN can model highly non-linear boundaries.

## Common Interview Traps
*   **Forgetting to scale data:** The #1 trap. Always mention standardizing features before applying KNN.
*   **Ignoring inference latency:** Suggesting KNN for a real-time web application with millions of users without mentioning approximate nearest neighbors (ANN, e.g., FAISS).
*   **Misunderstanding K=Even:** For binary classification, choosing an even K can result in ties. (Rule of thumb: use an odd K for binary classification).

## Python / SQL Syntax (if applicable)
```python
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

# Always scale data first!
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit and predict
knn = KNeighborsClassifier(n_neighbors=5, metric='minkowski', p=2) # p=2 is Euclidean
knn.fit(X_scaled, y)
predictions = knn.predict(X_new_scaled)
```

## Important Formula (if applicable)
*   **Euclidean Distance (L2):** $$d(\mathbf{p}, \mathbf{q}) = \sqrt{\sum_{i=1}^{n} (q_i - p_i)^2}$$
*   **Manhattan Distance (L1):** $$d(\mathbf{p}, \mathbf{q}) = \sum_{i=1}^{n} |q_i - p_i|$$
*   **Cosine Similarity:** $$S_C(\mathbf{A}, \mathbf{B}) = \frac{\mathbf{A} \cdot \mathbf{B}}{\|\mathbf{A}\|\|\mathbf{B}\|}$$

## 45-Second Interview Answer
"KNN is a non-parametric, lazy learning algorithm that classifies new data points based on the majority class of their 'K' closest neighbors in the feature space. Because it calculates distances—typically Euclidean or Manhattan—feature scaling is absolutely mandatory before using it. Its primary advantage is its simplicity and ability to model highly complex, non-linear boundaries. However, it suffers from high inference latency, high memory requirements, and the curse of dimensionality, making it unsuitable for very large or high-dimensional datasets unless we use Approximate Nearest Neighbor techniques like FAISS."

## Practice Questions:

### Q1:
You are building a content-based recommender system for a news application. You have converted the articles into TF-IDF vectors (high-dimensional text embeddings). You need to write a simple script to find the 2 most similar articles to a given target article using K-Nearest Neighbors.

**The Question:**
Which distance/similarity metric is best suited for this TF-IDF text data and why? Write a short Python snippet using scikit-learn to instantiate and fit a Nearest Neighbors model to retrieve the 2 nearest neighbors for 'Doc_A'.

In [18]:
# Data:
import pandas as pd
from sklearn.neighbors import NearestNeighbors

# Mock TF-IDF DataFrame representing 4 documents (A, B, C, D) and 4 vocabulary words
data = {
    'doc_id': ['Doc_A', 'Doc_B', 'Doc_C', 'Doc_D'],
    'word_AI': [0.8, 0.1, 0.0, 0.9],
    'word_ML': [0.7, 0.0, 0.1, 0.8],
    'word_Sports': [0.0, 0.9, 0.8, 0.1],
    'word_Food': [0.0, 0.1, 0.9, 0.0]
}
df = pd.DataFrame(data)

# Features only
X = df.drop('doc_id', axis=1)

**Answer:**
*   **Metric:** Cosine Similarity (or Cosine Distance).
*   **Why:** Cosine similarity measures the angle between two vectors, completely ignoring their magnitude. In text data, magnitude is largely driven by document length. We want a short article about "Machine Learning" and a long article about "Machine Learning" to be considered highly similar, which Euclidean distance would fail to do.

In [19]:
from sklearn.neighbors import NearestNeighbors

# Fit model using cosine metric. 
# We set n_neighbors=3 because querying a point in the training set 
# will always return the point itself as the closest match (distance 0).
nn_model = NearestNeighbors(n_neighbors=3, metric='cosine')
nn_model.fit(X)

# Query the first document (using iloc[[0]] to maintain a 2D array shape)
distances, indices = nn_model.kneighbors(X.iloc[[0]])

# indices[0][1:] will contain the indices of the 2 actual nearest neighbors

In [21]:
indices[0][1:]

array([3, 1])

**Interview Tip:** Always mention the "self-match" trap in KNN when querying points that already exist in your training/index set. Mentioning that you need $K+1$ neighbors proves you have actually built this in practice.

### Q2:
## Practice Question: KNN at Production Scale
**Question:** You have 50 million TF-IDF vectors. You need to return nearest neighbors in <50ms. Why will standard `scikit-learn` KNN fail, and how do you fix it?

**Answer:**
*   **Why it fails:** Standard KNN is an exact, "brute-force" algorithm. It has $O(N \times D)$ inference time. To find neighbors, it must load all 50M vectors into memory and calculate the exact distance from the query to every single vector. This will take seconds or minutes, failing the 50ms SLA.
*   **The Solution:** We must switch from Exact Nearest Neighbors to **Approximate Nearest Neighbors (ANN)**. 
*   **How it works:** We use a library like **FAISS** (Facebook AI Similarity Search) or a Vector Database (Pinecone, Milvus, Qdrant). These tools pre-build indexes (using techniques like HNSW or IVF) that group similar vectors together. At inference time, they only search the relevant "neighborhood" of vectors, reducing search time to milliseconds at the cost of a tiny fraction of accuracy.

**Interview Tip:** If an interviewer asks about RAG (Retrieval-Augmented Generation) in this context, clarify that ANN/Vector Databases are the engine powering the "R" (Retrieval) in RAG!